# YOLOv8 Pruning + QAT Pipeline

Full pipeline on Google Colab:

1. **Sparsity training** — L1 regularization on BN gamma
2. **Structured pruning** — Remove channels based on BN gamma threshold
3. **Finetune** — Recovery training on pruned model
4. **QAT** — Quantization-Aware Training (INT8 fake quantization)
5. **Export** — ONNX with Q/DQ nodes → TensorRT INT8 engine

## 0. Setup Environment

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Colab: {IN_COLAB}")
print(f"Python: {sys.version.split()[0]}")

import torch
print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
assert torch.cuda.is_available(), "GPU required!"

In [ ]:
# Mount Google Drive (dataset stored here)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Clone repo
REPO_URL = "https://github.com/NguyenTienAnh8365/Deploying-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices.git"
BRANCH = "pruned-QAT"
CLONE_DIR = Path("/content/yolov8-pruned-QAT")

if IN_COLAB:
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR)
    !git clone -b {BRANCH} --single-branch {REPO_URL} {CLONE_DIR}
    # Repo root contains yolov8-pruned-QAT/ subfolder, but on pruned-QAT branch
    # the working code is at repo root or inside a subfolder.
    # Adjust REPO to point to the folder containing prune.py and qat_pruned.py
    if (CLONE_DIR / "prune.py").exists():
        REPO = CLONE_DIR
    elif (CLONE_DIR / "yolov8-pruned-QAT" / "prune.py").exists():
        REPO = CLONE_DIR / "yolov8-pruned-QAT"
    else:
        # Search for prune.py
        hits = list(CLONE_DIR.rglob("prune.py"))
        assert hits, f"prune.py not found in {CLONE_DIR}"
        REPO = hits[0].parent
else:
    # Local: notebook is inside the repo
    REPO = Path.cwd().resolve()
    if not (REPO / "prune.py").exists():
        REPO = REPO / "yolov8-pruned-QAT"

assert (REPO / "prune.py").exists(), f"prune.py not found in {REPO}"
assert (REPO / "qat_pruned.py").exists(), f"qat_pruned.py not found in {REPO}"
print(f"REPO: {REPO}")

In [ ]:
# Install dependencies
!cd {REPO} && pip install -e . -q
!pip install --upgrade setuptools wheel -q
!pip install nvidia-pyindex -q
!pip install --no-cache-dir pytorch-quantization==2.1.2 --extra-index-url https://pypi.nvidia.com -q
!pip install "numpy<2.0" -q

# Verify
import ultralytics
import pytorch_quantization
print(f"ultralytics: {ultralytics.__version__}")
print(f"pytorch_quantization: {pytorch_quantization.__version__}")

## 1. Config

In [ ]:
import yaml

# ── Paths ─────────────────────────────────────────────────────────────
# Dataset: sua lai cho dung path tren Drive cua ban
DATASET_DRIVE = Path("/content/drive/MyDrive/dataset")  # <-- SUA NEU CAN
DATASET_LOCAL = Path("/content/dataset")

# Pretrained weight (FP32 best.pt tu fine-tune truoc do)
PRETRAINED_WEIGHT = Path("/content/drive/MyDrive/weights/best.pt")  # <-- SUA NEU CAN

# ── Model ─────────────────────────────────────────────────────────────
NC = 3
CLASS_NAMES = ["PAPER", "PLASTIC", "GLASS"]
MODEL_SIZE = "s"
MODEL_CFG_PATH = REPO / "ultralytics" / "cfg" / "models" / "v8" / "yolov8.yaml"
IMGSZ = 640
DEVICE = 0

# ── Directories ────────────────────────────────────────────────────────
WEIGHTS_DIR = REPO / "weights"
RUNS_DIR = REPO / "runs"
DATA_YAML = REPO / "data.yaml"
WEIGHTS_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)

# ── Workers ────────────────────────────────────────────────────────────
WORKERS = 4 if IN_COLAB else 0
CACHE = "ram" if IN_COLAB else "disk"

print(f"REPO:       {REPO}")
print(f"DATASET:    {DATASET_DRIVE}")
print(f"PRETRAINED: {PRETRAINED_WEIGHT}")

In [ ]:
# Copy dataset to local disk (faster I/O than Drive)
if IN_COLAB:
    assert DATASET_DRIVE.exists(), f"Dataset not found: {DATASET_DRIVE}"
    if DATASET_LOCAL.exists():
        shutil.rmtree(DATASET_LOCAL)
    shutil.copytree(DATASET_DRIVE, DATASET_LOCAL)
    print(f"Copied dataset -> {DATASET_LOCAL}")
else:
    DATASET_LOCAL = DATASET_DRIVE  # local: use directly

# Verify splits
for split in ["train", "val", "test"]:
    img_dir = DATASET_LOCAL / split / "images"
    lbl_dir = DATASET_LOCAL / split / "labels"
    assert img_dir.exists(), f"Missing: {img_dir}"
    assert lbl_dir.exists(), f"Missing: {lbl_dir}"
    print(f"{split:5s}: {len(list(img_dir.glob('*'))):5d} images")

# Write data.yaml
data_cfg = {
    "train": str(DATASET_LOCAL / "train" / "images"),
    "val":   str(DATASET_LOCAL / "val" / "images"),
    "test":  str(DATASET_LOCAL / "test" / "images"),
    "nc": NC,
    "names": CLASS_NAMES,
}
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(f"\nWrote: {DATA_YAML}")
print(DATA_YAML.read_text())

In [ ]:
# Copy pretrained weight to repo weights dir
assert PRETRAINED_WEIGHT.exists(), f"Pretrained weight not found: {PRETRAINED_WEIGHT}"
LOCAL_PRETRAINED = WEIGHTS_DIR / "original.pt"
if not LOCAL_PRETRAINED.exists():
    shutil.copy2(PRETRAINED_WEIGHT, LOCAL_PRETRAINED)
print(f"Pretrained: {LOCAL_PRETRAINED} ({LOCAL_PRETRAINED.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Helper
def run(cmd):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)

## 2. Step 1 — Sparsity Training

Train with L1 regularization on BN gamma (`sr` parameter) to push unimportant channels toward zero.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(LOCAL_PRETRAINED))
model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=IMGSZ,
    batch=32,
    sr=1e-2,
    lr0=1e-3,
    patience=50,
    workers=WORKERS,
    cache=CACHE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="train-sparsity",
    exist_ok=True,
)

In [ ]:
SPARSITY_WEIGHT = RUNS_DIR / "train-sparsity" / "weights" / "last.pt"
assert SPARSITY_WEIGHT.exists(), f"Not found: {SPARSITY_WEIGHT}"
print(f"Step 1 done: {SPARSITY_WEIGHT}")

## 3. Step 2 — Structured Pruning

Remove channels where BN gamma < threshold. Output: `pruned.pt` with `maskbndict`.

In [ ]:
PRUNE_RATIO = 0.05

run([
    sys.executable, "prune.py",
    "--weights", str(SPARSITY_WEIGHT),
    "--cfg", str(MODEL_CFG_PATH),
    "--model-size", MODEL_SIZE,
    "--prune-ratio", str(PRUNE_RATIO),
    "--save-dir", str(WEIGHTS_DIR),
])

In [ ]:
import glob

pruned_files = sorted(glob.glob(str(WEIGHTS_DIR / "pruned*.pt")))
assert pruned_files, f"No pruned weight found in {WEIGHTS_DIR}"
PRUNED_WEIGHT = Path(pruned_files[-1])

ckpt = torch.load(PRUNED_WEIGHT, map_location="cpu", weights_only=False)
assert "maskbndict" in ckpt, "Invalid pruned checkpoint: missing maskbndict"
params = sum(p.numel() for p in ckpt["model"].parameters())
print(f"Step 2 done: {PRUNED_WEIGHT}")
print(f"Pruned params: {params:,}")

## 4. Step 3 — Finetune Pruned Model

Recovery training to restore accuracy after pruning.

In [ ]:
model = YOLO(str(PRUNED_WEIGHT))
model.train(
    data=str(DATA_YAML),
    epochs=150,
    imgsz=IMGSZ,
    batch=32,
    sr=0.0,
    finetune=True,
    patience=50,
    workers=WORKERS,
    cache=CACHE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="train-finetune",
    exist_ok=True,
)

In [ ]:
best = RUNS_DIR / "train-finetune" / "weights" / "best.pt"
last = RUNS_DIR / "train-finetune" / "weights" / "last.pt"
FINETUNE_WEIGHT = best if best.exists() else last
assert FINETUNE_WEIGHT.exists(), f"Not found: {FINETUNE_WEIGHT}"
print(f"Step 3 done: {FINETUNE_WEIGHT}")

## 5. Step 4 — QAT on Pruned Model

Quantization-Aware Training with INT8 fake quantization.

Note: `--lr0 1e-3` → trainer auto-scales to `1e-5` (divides by 100).

In [ ]:
run([
    sys.executable, "qat_pruned.py",
    "--pruned-checkpoint", str(PRUNED_WEIGHT),
    "--pretrained-weight", str(FINETUNE_WEIGHT),
    "--data-config", str(DATA_YAML),
    "--epochs", "10",
    "--imgsz", str(IMGSZ),
    "--batch", "16",
    "--device", str(DEVICE),
    "--workers", str(WORKERS),
    "--cache", CACHE,
    "--lr0", "1e-3",
    "--freeze", "10",
    "--optimizer", "AdamW",
    "--patience", "5",
    "--recalib-every", "1",
    "--project", str(RUNS_DIR / "qat-pruned"),
    "--name", "train",
    "--exist-ok",
    "--plots",
])

In [ ]:
qat_best = RUNS_DIR / "qat-pruned" / "train" / "weights" / "best.pt"
qat_last = RUNS_DIR / "qat-pruned" / "train" / "weights" / "last.pt"
QAT_WEIGHT = qat_best if qat_best.exists() else qat_last
assert QAT_WEIGHT.exists(), f"Not found: {QAT_WEIGHT}"
print(f"Step 4 done: {QAT_WEIGHT}")

## 6. Step 5 — Export ONNX + TensorRT Engine

Export ONNX with Q/DQ nodes. Build `.engine` if `trtexec` is available.

In [ ]:
run([
    sys.executable, "qat_pruned_export.py",
    "--pruned-checkpoint", str(PRUNED_WEIGHT),
    "--weight", str(QAT_WEIGHT),
])

onnx_files = sorted(QAT_WEIGHT.parent.glob("*.tensorrt.onnx"))
if not onnx_files:
    onnx_files = sorted(QAT_WEIGHT.parent.glob("*.onnx"))
assert onnx_files, f"No ONNX found in {QAT_WEIGHT.parent}"
ONNX_PATH = onnx_files[-1]
print(f"ONNX: {ONNX_PATH} ({ONNX_PATH.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Build TensorRT engine (optional — requires trtexec)
ENGINE_PATH = None
trtexec = shutil.which("trtexec")

if trtexec:
    ENGINE_PATH = ONNX_PATH.with_suffix(".engine")
    try:
        run([trtexec, f"--onnx={ONNX_PATH}", f"--saveEngine={ENGINE_PATH}", "--int8", "--fp16"])
        print(f"Engine: {ENGINE_PATH} ({ENGINE_PATH.stat().st_size / 1e6:.1f} MB)")
    except Exception as e:
        print(f"TensorRT build failed: {e}")
        ENGINE_PATH = None
else:
    print("trtexec not found — skip engine build")

## 7. Evaluation (Optional)

Compare mAP across pipeline stages.

In [ ]:
from ultralytics import YOLO

def eval_map(weight, label):
    m = YOLO(str(weight))
    r = m.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=16, device=DEVICE, workers=WORKERS, verbose=False)
    print(f"{label:<25} mAP50={float(r.box.map50):.4f}  mAP50-95={float(r.box.map):.4f}")

eval_map(LOCAL_PRETRAINED, "FP32 Original")
eval_map(FINETUNE_WEIGHT, "Pruned + Finetune")
eval_map(QAT_WEIGHT, "Pruned + QAT")
if ENGINE_PATH and ENGINE_PATH.exists():
    eval_map(ENGINE_PATH, "TensorRT INT8")

In [ ]:
# Inference preview
import cv2
import matplotlib.pyplot as plt

preview_w = ENGINE_PATH if (ENGINE_PATH and ENGINE_PATH.exists()) else QAT_WEIGHT
model = YOLO(str(preview_w))

test_dir = DATASET_LOCAL / "test" / "images"
if not test_dir.exists():
    test_dir = DATASET_LOCAL / "val" / "images"
imgs = sorted(test_dir.glob("*"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, p in zip(axes.flat, imgs):
    r = model.predict(str(p), conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(p.name)
    ax.axis("off")
plt.suptitle(f"Preview — {preview_w.name}", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Save Results to Drive (Optional)

In [ ]:
if IN_COLAB:
    SAVE_DIR = Path("/content/drive/MyDrive/pruned-qat-results")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    for f in [PRUNED_WEIGHT, FINETUNE_WEIGHT, QAT_WEIGHT, ONNX_PATH]:
        if f and f.exists():
            shutil.copy2(f, SAVE_DIR / f.name)
            print(f"Saved: {f.name}")
    if ENGINE_PATH and ENGINE_PATH.exists():
        shutil.copy2(ENGINE_PATH, SAVE_DIR / ENGINE_PATH.name)
        print(f"Saved: {ENGINE_PATH.name}")

    print(f"\nAll results saved to: {SAVE_DIR}")
else:
    print("Not on Colab — results are in:", RUNS_DIR)